# ML-03 — Frame Your Lane as an ML Task

## 1. My lane as an ML task (type)

**Provisional lane: Lane 2 — Refresh / Content Opportunity Scoring.**

I frame this as a **ranking/scoring** problem. The decision is: when review capacity is limited, which content pages should a reviewer inspect first? The output is a priority score and ranked review queue. Ranking fits because the team needs an ordered shortlist rather than a yes/no prediction for every page. This is decision support for a human reviewer, not an automatic final content decision.

In [1]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print("Task type: ranking / scoring")
print("Rows loaded:", len(df))
print("Unique content pages:", df["content_id"].nunique())

Task type: ranking / scoring
Rows loaded: 30,000
Unique content pages: 30,000


## 2. Target or proxy

For the starter dataset, the immediate **proxy** is `is_declining_label`, which is derived from the current `trend_direction` signal. I would use this only as a starter proxy, not as a claim about what will happen in the future.

For a stronger version, I would define an observed future outcome in a later time window and use only earlier-window features. This keeps the feature window separate from the target window and reduces leakage risk. The goal is not to predict that a page is "bad"; it is to identify pages that deserve earlier human review.

In [2]:
df["is_declining_proxy"] = df["trend_direction"].eq("down")
print(f"Starter proxy positive rate: {df['is_declining_proxy'].mean() * 100:.1f}%")

Starter proxy positive rate: 54.2%


## 3. Success metric

My primary success metric is **Precision@K**, with K chosen to represent realistic review capacity, such as K=20 or K=50. If the team can review only the first K pages, Precision@K tells us how many selected pages match the chosen positive target/proxy. This is more aligned with the decision than overall accuracy. I would compare a learned ranking with a transparent fixed-rule baseline before claiming that ML adds value.

In [3]:
K = 50
print(f"Primary decision-aligned metric: Precision@{K}")

Primary decision-aligned metric: Precision@50


## 4. The unit of analysis, as a real dataframe

**One row = one pseudonymized content page.** The starter data contains 30,000 page-level rows. `content_id` identifies the page, while `client_id` can be used for grouping or split design rather than as a predictive feature. The dataframe below shows the page-level unit and observable signals.

In [4]:
lane_df = df[["content_id", "client_id", "impressions_90d", "clicks_90d", "sessions_90d", "content_age_days", "trend_direction"]].copy()
print("Shape:", lane_df.shape)
print("One row = one content page")
print(lane_df.head(8).to_string(index=False))

Shape: (30000, 7)
One row = one content page
          content_id        client_id  impressions_90d  clicks_90d  sessions_90d  content_age_days trend_direction
content_304f48230142 client_f369cb89fc             3803          29            17               187            down
content_a1fb4e703a9e client_4e07408562            15320           7             9               445            down
content_9aa793d4d895 client_7f2253d7e            12581          11            11               141            down
content_331d6c4de07b client_19581e27de            11751          58            78               463          stable
content_d99b7a2d90ca client_3fdba35f04            19140          24           145               263            down
content_d4084a4bc775 client_f369cb89fc             3970           1             5               147            down
content_9a34b442b552 client_8722616204               20           0             1                90            down
content_a63219c6e95a client_1

## 5. Why ML beats a fixed rule here

A fixed rule is an important baseline, but several observable signals can interact. Impressions, clicks, sessions, page age, and other content/search signals may combine differently across pages. Writing every useful interaction as hand-built thresholds can become brittle.

However, I do **not** assume ML is better. ML earns its place only if it reliably improves the ranking over a transparent rule under honest, leakage-safe validation. If the rule performs just as well, the rule may be the better solution because it is simpler and easier to explain.

In [5]:
print("Baseline: transparent fixed-rule ranking")
print("Candidate: learned ranking model")
print("ML earns its place only if it improves Precision@K under honest validation.")

Baseline: transparent fixed-rule ranking
Candidate: learned ranking model
ML earns its place only if it improves Precision@K under honest validation.


## Self-check

- [x] Task type: ranking/scoring
- [x] Target/proxy named and limitation explained
- [x] Success metric: Precision@K
- [x] Real dataframe shown; one row = one content page
- [x] Output supports a real content action through a prioritized human review queue
- [x] ML is not assumed to be better; it must beat a transparent baseline
- [x] Claims are careful: observed, proxy, directional, decision-support
- [x] Notebook is under `work/notebooks/w02_ml_task_framing.ipynb`